Imports

In [1]:
# set working directory to this jupyter notebook file's location
import os
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from plotnine import *
import networkx as nx

import numpy as np
import networkx as nx
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter
import random

from sklearn.metrics import roc_auc_score, roc_curve, auc


# set working directory to this jupyter notebook file's location
if '__file__' in locals():
    path = os.path.join(os.getcwd(), os.path.dirname(__file__))
else:
    path = os.getcwd()
os.chdir(path)
sys.path.append(path)

1. (50 pts total) Predicting missing node labels. Let $G = (V, E)$ be a graph and let $\sim x$ be a vector of categorical node attributes. Recall that labels exhibit assortative mixing if labels $x_i, x_j$ are more likely to be similar (or the same) if $(i, j) \in E$ than if not. When this is true, we can use the “local smoothing” heuristic from the lectures to make a simple unsupervised guess about any particular missing label.

Go to the Index of Complex Networks at icon.colorado.edu and obtain the following:
- ICON entry: “Norwegian Boards of Directors (2002-2011, projection)” 
    - network: net1m_2011-08-01
    - metadata: data_people (gender variable)

- ICON entry: “Malaria var DBLa HVR networks”
    - network: HVR_5
    - metadata: metadata_CysPoLV

(a) (50 pts) Design and carry out an experiment to systematically evaluate the local smoothing heuristic as a function of $\alpha$, the fraction of node labels we observed for some $G$.
- Implement the local smoothing heuristic as described in the lecture notes; don’t forget to default to the baseline when necessary, and break ties randomly.
- Let $\alpha$ vary between 0 and 1 in ﬁxed increments, e.g., $\Delta \alpha = 0.02$, and measure the average accuracy (ACC) over some number of repetitions at each $\alpha$. (Hint: more repetitions makes a smoother curve.)
- Plot the average ACC functions for the two networks on a single nice ﬁgure.
- Discuss: (i) How are the accuracy curves similar/diﬀerent between the two networks (be sure to at least discuss the performance in the very low, mid-range, and very high $\alpha$ ranges)? (ii) What if anything do you learn about the local smoothing heuristic from this experiment? And, (iii) What if any insights do you gain about the structure of these networks from the shape of these curves?
- Derive mathematically the expected accuracy for the baseline predictor (guessing a missing label uniformly at random from $\sim x_o$); calculate the baseline expectation for the two networks, and comment on how the results of your experiment compare to this baseline

In [2]:
malaria_network_path = "files/HVR_5.txt"
malaria_metadata_path = "files/metadata_CysPoLV.txt"

bod_edges_path = "files/net1m_2011-08-01.csv/edges.csv"
bod_nodes_path = "files/net1m_2011-08-01.csv/nodes.csv"

# Read network function
def read_network(edge_path, node_path=None, delimiter=",", comment_char="#"):
    """
    Reads a network from an edge list file (TXT or CSV).

    Parameters:
        edge_path (str): Path to the edge list file.
        node_path (str, optional): Path to the node metadata file (if available).
        delimiter (str, optional): Delimiter used in the edge list file (default: ",").
        comment_char (str, optional): Character used to denote comments in the file (default: "#").

    Returns:
        G (networkx.Graph): The constructed graph.
        node_metadata (pd.DataFrame or None): Node metadata if provided, else None.
    """
    G = nx.Graph()

    # Read edge list (TXT or CSV)
    try:
        if edge_path.endswith(".csv"):
            df = pd.read_csv(edge_path, comment=comment_char, delimiter=delimiter)
            G.add_edges_from(df.values)
        else:
            with open(edge_path, "r") as file:
                for line in file:
                    if not line.startswith(comment_char):
                        node1, node2 = map(int, line.strip().split(delimiter))
                        G.add_edge(node1, node2)
    except Exception as e:
        print(f"Error reading {edge_path}: {e}")
        return None, None

    # Read node metadata if provided
    node_metadata = None
    if node_path:
        try:
            node_metadata = pd.read_csv(node_path, sep=delimiter, index_col=False, comment=comment_char, header=None)
        except Exception as e:
            print(f"Error reading {node_path}: {e}")

    return G, node_metadata

# Read malaria network
malaria_G, malaria_metadata = read_network(
    malaria_network_path, malaria_metadata_path, delimiter=","
)

# Read BOD network
bod_G, bod_metadata = read_network(
    bod_edges_path, bod_nodes_path, delimiter=","
)

malaria_metadata.columns = ["CysPoLV"]
bod_metadata.columns = ["index", "vid", "name", "gender", "_pos"]

# print("Malaria network:")
# print(malaria_metadata.head())
# print("\nBOD network:")
# print(bod_metadata.head())

In [3]:
def remove_labels(node_labels, alpha, seed=None):
    """
    Randomly removes labels from a given fraction (1 - alpha) of nodes.

    Parameters:
        node_labels (dict or pd.Series): Mapping of node IDs to labels.
        alpha (float): Fraction of labels to keep (0 ≤ alpha ≤ 1).
        seed (int, optional): Random seed for reproducibility.

    Returns:
        dict or pd.Series: The modified labels with some set to None.
    """
    if seed is not None:
        random.seed(seed)
        np.random.seed(seed)

    nodes = (
        list(node_labels.keys())
        if isinstance(node_labels, dict)
        else node_labels.index.tolist()
    )
    num_keep = int(alpha * len(nodes))  # Number of labels to keep

    # Randomly select which nodes to keep
    keep_nodes = set(random.sample(nodes, num_keep))

    # Assign None to removed labels
    if isinstance(node_labels, dict):
        new_labels = {
            node: (node_labels[node] if node in keep_nodes else None) for node in nodes
        }
    else:  # Assume it's a pandas Series
        new_labels = node_labels.copy()
        new_labels[~new_labels.index.isin(keep_nodes)] = None

    # Ensure the new_labels dtype is the same as the input, but allow for None/NaN in the case of integer columns
    if isinstance(node_labels, pd.Series):
        if pd.api.types.is_integer_dtype(node_labels):
            # Convert to float to allow for NaN values
            new_labels = new_labels.astype(float)
        else:
            new_labels = new_labels.astype(node_labels.dtype)

    return new_labels


def baseline_predictor(node_labels, seed=None):
    """
    Fills in missing labels by randomly assigning them based on the observed label distribution.

    Parameters:
        node_labels (dict or pd.Series): Mapping of node IDs to labels (some missing).
        seed (int, optional): Random seed for reproducibility.

    Returns:
        dict or pd.Series: Labels with missing values filled in using baseline prediction.
    """
    if seed is not None:
        random.seed(seed)
        np.random.seed(seed)

    # Extract observed (non-missing) labels
    if isinstance(node_labels, pd.Series):
        observed_labels = node_labels.dropna().to_numpy()  # Ensure it's an array
    else:
        observed_labels = np.array(
            [v for v in node_labels.values() if v is not None]
        )  # Dict case

    # Get label distribution (frequency counts)
    label_counts = Counter(observed_labels)  # Now observed_labels is an array
    label_list, label_probs = zip(
        *[
            (label, count / sum(label_counts.values()))
            for label, count in label_counts.items()
        ]
    )

    # Fill in missing labels based on this distribution
    missing_nodes = (
        node_labels.index[node_labels.isna()].tolist()
        if isinstance(node_labels, pd.Series)
        else [node for node, label in node_labels.items() if label is None]
    )

    # Choose the appropriate data structure for the output (matching input type)
    if isinstance(node_labels, pd.Series):
        filled_labels = node_labels.copy()
        for node in missing_nodes:
            filled_labels[node] = np.random.choice(label_list, p=label_probs)
    else:  # dict case
        filled_labels = node_labels.copy()
        for node in missing_nodes:
            filled_labels[node] = np.random.choice(label_list, p=label_probs)

    return filled_labels


def evaluate_prediction_accuracy(original_labels, missing_labels, predicted_labels):
    """
    Evaluates the accuracy of the baseline predictor by comparing predicted labels
    to the original labels before removal.

    Parameters:
        original_labels (dict or pd.Series): The true labels before removal.
        predicted_labels (dict or pd.Series): The labels predicted by the baseline predictor.

    Returns:
        float: Accuracy score (0 to 1).
    """
    # Ensure both inputs are Pandas Series for easy comparison
    if isinstance(original_labels, dict):
        original_labels = pd.Series(original_labels)
    if isinstance(predicted_labels, dict):
        predicted_labels = pd.Series(predicted_labels)

    # Identify nodes where labels were originally removed (i.e., missing before prediction)
    missing_labels_mask = missing_labels.isna()

    # Extract only the labels that were originally missing
    true_values = original_labels[missing_labels_mask]
    predicted_values = predicted_labels[missing_labels_mask]

    # Compute accuracy
    correct_predictions = (true_values == predicted_values).sum()
    total_predictions = len(true_values)

    accuracy = correct_predictions / total_predictions if total_predictions > 0 else 0
    return accuracy


def local_smoothing_predictor(node_labels, graph, seed=None):
    """
    Predicts missing node labels using local smoothing heuristics, based on neighboring nodes' labels in a graph.

    Parameters:
        node_labels (dict or pd.Series): Mapping of node IDs to labels (some missing).
        graph (networkx.Graph): The graph representing the relationships between nodes.
        seed (int, optional): Random seed for reproducibility.

    Returns:
        dict or pd.Series: Labels with missing values filled using local smoothing.
    """
    if seed is not None:
        np.random.seed(seed)

    # Ensure node_labels is in a pandas Series format
    if isinstance(node_labels, dict):
        node_labels = pd.Series(node_labels)

    # Identify missing node labels while preserving node IDs
    missing_nodes = node_labels[node_labels.isna()].index.tolist()

    # Create a copy to fill in missing values
    smoothed_labels = node_labels.copy()

    # Iterate over missing nodes to apply local smoothing
    for node in missing_nodes:
        if node in graph:
            neighbors = list(
                graph.neighbors(node)
            )  # Get neighbors while keeping original node IDs
            neighbor_labels = [
                node_labels.loc[neighbor]
                for neighbor in neighbors
                if neighbor in node_labels and not pd.isna(node_labels.loc[neighbor])
            ]

            if neighbor_labels:  # If there are valid labeled neighbors
                smoothed_labels.loc[node] = np.random.choice(neighbor_labels)

    # Handle any remaining missing labels by assigning them based on global label distribution
    remaining_missing_nodes = smoothed_labels[smoothed_labels.isna()].index.tolist()
    if remaining_missing_nodes:
        # Extract observed (non-missing) labels
        observed_labels = node_labels.dropna().to_numpy()

        # Compute label distribution
        label_counts = Counter(observed_labels)
        label_list, label_probs = zip(
            *[
                (label, count / sum(label_counts.values()))
                for label, count in label_counts.items()
            ]
        )

        # Assign missing nodes using this distribution
        for node in remaining_missing_nodes:
            smoothed_labels.loc[node] = np.random.choice(label_list, p=label_probs)

    # Return the smoothed labels in the same format as the input
    return (
        smoothed_labels.to_dict() if isinstance(node_labels, dict) else smoothed_labels
    )


In [8]:
# create a list from 0 to 1 by increments of 0.02
alphas = np.arange(0.02, 1.02, 0.02)
results = {}

for alpha in alphas:

    # Remove labels from malaria network
    malaria_labels = malaria_metadata["CysPoLV"].astype(float)
    malaria_labels_modified = remove_labels(malaria_labels, alpha, seed=42)

    # Remove labels from BOD network
    bod_labels = bod_metadata.set_index("vid")["gender"]
    bod_labels_modified = remove_labels(bod_labels, alpha, seed=42)

    # Apply baseline predictor to both networks
    malaria_filled = baseline_predictor(malaria_labels_modified, seed=42)
    bod_filled = baseline_predictor(bod_labels_modified, seed=42)

    malaria_smoothed_labels = local_smoothing_predictor(malaria_labels_modified, malaria_G, seed=42)
    bod_smoothed_labels = local_smoothing_predictor(bod_labels_modified, bod_G, seed=42)

    # Evaluate prediction accuracy
    malaria_accuracy = evaluate_prediction_accuracy(
        malaria_labels, malaria_labels_modified, malaria_filled
    )
    bod_accuracy = evaluate_prediction_accuracy(bod_labels, bod_labels_modified, bod_filled)

    # evaluate the smoothing predictor accuracy
    malaria_smoothed_accuracy = evaluate_prediction_accuracy(
        malaria_labels, malaria_labels_modified, malaria_smoothed_labels
    )
    bod_smoothed_accuracy = evaluate_prediction_accuracy(
        bod_labels, bod_labels_modified, bod_smoothed_labels
    )

    results[alpha] = {
        "malaria_baseline": malaria_accuracy,
        "malaria_smoothed": malaria_smoothed_accuracy,
        "bod_baseline": bod_accuracy,
        "bod_smoothed": bod_smoothed_accuracy,
    }

# can we now make a ggplot using plotnine to plot the results?]
results_df = pd.DataFrame(results).T.reset_index()
results_df = results_df.melt(id_vars=["index"], var_name="network", value_name="accuracy")
results_df["method"] = results_df["network"].str.split("_", expand=True)[1]


# Convert results dictionary to DataFrame
results_df = pd.DataFrame(results).T.reset_index()
results_df.rename(
    columns={"index": "alpha"}, inplace=True
)  # Ensure alpha is correctly named

# Reshape for ggplot using melt (apply only ONCE)
results_df = results_df.melt(
    id_vars=["alpha"], var_name="network_method", value_name="accuracy"
)

# Extract network and method
results_df[["network", "method"]] = results_df["network_method"].str.split(
    "_", expand=True
)

# === PLOT FOR BOD NETWORK ===
bod_plot = (
    ggplot(
        results_df[results_df["network"] == "bod"],
        aes(x="alpha", y="accuracy", color="method"),
    )
    + geom_line(size=1.2)
    + scale_color_manual(values=["#E63946", "#1D3557"], labels=["Baseline", "Smoothed"])
    + labs(
        x="Fraction of Labels Removed",
        y="Prediction Accuracy",
        title="BOD Network Accuracy",
    )
    + theme_minimal()
    + theme(legend_title=element_blank())
)

# === PLOT FOR MALARIA NETWORK ===
malaria_plot = (
    ggplot(
        results_df[results_df["network"] == "malaria"],
        aes(x="alpha", y="accuracy", color="method"),
    )
    + geom_line(size=1.2)
    + scale_color_manual(values=["#E63946", "#1D3557"], labels=["Baseline", "Smoothed"])
    + labs(
        x="Fraction of Labels Removed",
        y="Prediction Accuracy",
        title="Malaria Network Accuracy",
    )
    + theme_minimal()
    + theme(legend_title=element_blank())
)

# Create a single plot with different line types for malaria and BOD
combined_plot = (
    ggplot(results_df, aes(x="alpha", y="accuracy", color="method", linetype="network"))
    + geom_line(size=1.2)
    + scale_color_manual(values=["#E63946", "#1D3557"], labels=["Baseline", "Smoothed"])
    + scale_linetype_manual(values=["solid", "dashed"], labels=["Malaria", "Board of \nDirectors"])
    + labs(
        x="Fraction of Labels Removed",
        y="Prediction Accuracy",
        title="Prediction Accuracy Across Networks",
    )
    + theme_minimal()
    + theme(legend_title=element_blank())
)

combined_plot.save("figures/combined_plot.png")

# bod_plot.save("figures/bod_prediction_node_attributes.png")
# malaria_plot.save("figures/malaria_prediction_node_attributes.png")

/opt/anaconda3/envs/networks/lib/python3.12/site-packages/plotnine/ggplot.py:615: PlotnineWarning: Saving 6.4 x 4.8 in image.
/opt/anaconda3/envs/networks/lib/python3.12/site-packages/plotnine/ggplot.py:616: PlotnineWarning: Filename: figures/combined_plot.png


(b) (25 pts extra credit) Design and carry out an experiment to evaluate the local smoothing heuristic as a function of how randomized the network is. That is, keep the fraction of observed labels $\alpha$ ﬁxed, and instead vary the fraction β of the network’s edges that are randomized. Test the hypothesis that as the network structure becomes increasingly randomized, the local smoothing heuristic’s accuracy degrades toward the baseline predictor.

For this experiment, set $\alpha = 0.8$, and use the removed $20\%$ of the node labels as the “test” set for calculating accuracy (ACC). Use the double-edge swap algorithm to parameterize the extent to which the structure is randomized by deﬁning $\beta = \frac{r}{2m}$, where r is the number of double edge swaps that have been applied to the original network $G$.
- Decide on a set of values spanning $r \in [0, 2m]$ that show oﬀ the overall pattern of how the local smoothing heuristic’s accuracy varies from $\beta = 0$ (fully empirical $G$) to $\beta = 1$ (fully randomized $G$).
- Run your experiment to measure the average accuracy for the 80/20 split of node labels as a function of $\beta$, over some number of repetitions for a particular value $\beta$. (Hint: more repetitions makes a smoother curve.)
- Make one nice ﬁgure showing accuracy vs. $\beta$ curves of the two networks. Indicate on the ﬁgure the expected baseline accuracy on the original network $G$ (e.g., with a horizontal line).
- Discuss: (i) To what degree do the accuracy curves diﬀer between these networks? (ii) How does the accuracy curve compare to the baseline on the original graph $G$? And, (iii) What if any insights does this experiment give you about the structure of these networks from the shape of these curves?

In [ ]:
def randomize_graph(G, beta):
    """
    Randomizes the edges of G using the double-edge swap method.
    The amount of randomization is controlled by beta.
    """
    m = G.number_of_edges()
    r = int(2 * m * beta)  # Compute number of swaps
    G_randomized = G.copy()
    nx.double_edge_swap(G_randomized, nswap=r, max_tries=10 * r)
    return G_randomized


def accuracy_plot_random_beta(dataframe, baseline_malaria, baseline_bod):
    """Creates a plot of prediction accuracy vs. network randomization
    
    Args:
        dataframe (pd.DataFrame): A DataFrame with columns ["beta", "network", "accuracy"]

    Returns:
        ggplot: A plotnine object representing the accuracy plot
    """
    # Plot accuracy vs. beta
    plot = (
        ggplot(dataframe, aes(x="beta", y="accuracy", color="network"))
        + geom_line(size=1.2)
        + geom_hline(yintercept=baseline_malaria, linetype="dashed", color="#E63946")
        + geom_hline(yintercept=baseline_bod, linetype="dashed", color="#1D3557")
        + annotate(
            "text",
            x=0.8,
            y=baseline_malaria + 0.01,
            label="Malaria Baseline",
            color="#E63946",
            size=8,
        )
        + annotate(
            "text",
            x=0.8,
            y=baseline_bod + 0.01,
            label="Board of Directors Baseline",
            color="#1D3557",
            size=8,
        )
        + labs(
            x=r"Fraction of Edges Randomized ($\beta$)",
            y="Prediction Accuracy",
            title="Prediction Accuracy vs. Network Randomization",
            color="Network",
        )
        + scale_color_manual(values=["#E63946", "#1D3557"])
        + theme_minimal()
        + scale_color_manual(
            values={"malaria": "#E63946", "bod": "#1D3557"},
            labels={"malaria": "Malaria", "bod": "Board of Directors"},
        )
    )

    return plot

# Define experiment parameters
betas = np.linspace(0, 1, 10)  # 10 points from 0 (original) to 1 (fully randomized)
num_repetitions = 5  # More repetitions = smoother curve
alpha = 0.8  # 80% observed labels

results = []

for beta in betas:
    for _ in range(num_repetitions):
        # Randomize malaria and BOD networks
        malaria_G_random = randomize_graph(malaria_G, beta)
        bod_G_random = randomize_graph(bod_G, beta)

        # Remove labels
        malaria_labels = malaria_metadata["CysPoLV"].astype(float)
        malaria_labels_modified = remove_labels(malaria_labels, alpha, seed=42)

        bod_labels = bod_metadata.set_index("vid")["gender"]
        bod_labels_modified = remove_labels(bod_labels, alpha, seed=42)

        # Apply local smoothing predictor
        malaria_smoothed_labels = local_smoothing_predictor(
            malaria_labels_modified, malaria_G_random, seed=42
        )
        bod_smoothed_labels = local_smoothing_predictor(
            bod_labels_modified, bod_G_random, seed=42
        )

        # Compute accuracy
        malaria_accuracy = evaluate_prediction_accuracy(
            malaria_labels, malaria_labels_modified, malaria_smoothed_labels
        )
        bod_accuracy = evaluate_prediction_accuracy(
            bod_labels, bod_labels_modified, bod_smoothed_labels
        )

        results.append(
            {"beta": beta, "network": "malaria", "accuracy": malaria_accuracy}
        )
        results.append({"beta": beta, "network": "bod", "accuracy": bod_accuracy})

# Convert results to DataFrame
results_df = pd.DataFrame(results)

# Compute baseline accuracy (original graph, beta=0)
baseline_malaria = results_df[results_df["beta"] == 0][
    results_df["network"] == "malaria"
]["accuracy"].mean()
baseline_bod = results_df[results_df["beta"] == 0][results_df["network"] == "bod"][
    "accuracy"
].mean()

# Plot accuracy vs. beta
accuracy_plot = accuracy_plot_random_beta(results_df, baseline_malaria, baseline_bod)

accuracy_plot.save("figures/accuracy_vs_randomization.png")

/var/folders/f2/lbj2f9h979b75rj33ps9wq780000gn/T/ipykernel_85497/3569895129.py:104: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
/var/folders/f2/lbj2f9h979b75rj33ps9wq780000gn/T/ipykernel_85497/3569895129.py:107: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
/opt/anaconda3/envs/networks/lib/python3.12/site-packages/plotnine/scales/scales.py:48: PlotnineWarning: Scale for 'color' is already present.
Adding another scale for 'color',
which will replace the existing scale.

/opt/anaconda3/envs/networks/lib/python3.12/site-packages/plotnine/ggplot.py:615: PlotnineWarning: Saving 6.4 x 4.8 in image.
/opt/anaconda3/envs/networks/lib/python3.12/site-packages/plotnine/ggplot.py:616: PlotnineWarning: Filename: figures/accuracy_vs_randomization.png


2. (50 pts) Predicting missing edges.

Recall the deﬁnitions of the degree product (DP) and Jaccard coeﬃcient (JC) link predictors from the lecture notes. To these, add the shortest path (SP) predictor, deﬁned as follows. Let
$\sigma(i, j)$ be the length of a geodesic path between $i$ and $j$. Then, the SP predictor is deﬁned as $score(i, j) = \frac{1}{\sigma}(i, j) + \epsilon$, where $\epsilon$ is a small amount of random noise. (Recall that we deﬁne $\sigma(i, j) = \inf$ if there is not path from $i$ to $j$.)

(a) (50 pts) Unsupervised link prediction.
- Implement the DP, JC, and SP score functions, each of which takes as input a node pair $i, j$ and an observed graph $G'$, and returns its respective score for that pair.
- Deﬁne accuracy as the AUC, and implement a function that takes as input the completed table over the set of potential missing links $X$ (see lecture notes), and calculates the corresponding AUC. (Hint: you can use one table with multiple score columns, one per predictor, which can then be row-sorted by a particular column to get a version you can use to calculate that predictor’s AUC.)
- Design and carry out a numerical experiment, using the same two networks as in question 1, in which you measure the accuracy of these three heuristics as a function of the fraction $f \in (0, 1)$ of the edges that are observed. (You will need to write a function to generate an observed graph $G'$, given $G$ and choice of $f$.)
- Let f vary in ﬁxed increments, e.g., $\Delta f = 0.05$, and measure the average AUC for each of the three heuristics at each $f$. (Hint: averaging over more repetitions at a choice of $f$ makes for smoother curves.)
- First, make a pair of ﬁgures (one for each network) that plot the corresponding accuracy curves for the 3 predictors; include a line at AUC = 0.5 as a reference.
- Second, make a nice ﬁgure showing all three full ROC curves for one of your networks, for $f = 0.8$. Indicate which network you chose.
- Discuss: (i) Why does one predictor perform much better than others when $f$ is very small, and worse than others when $f$ is larger? (ii) What if anything does the diﬀerence in performance across the two networks tell you about how those networks’ structures diﬀer? And, (iii) what do their relative shapes of the ROC curves imply about the accuracy of these algorithms?

In [16]:
def degree_product(G, i, j):
    """
    Computes the Degree Product (DP) for a given node pair (i, j).

    Parameters:
        G (networkx.Graph): The observed graph.
        i (int): Node i.
        j (int): Node j.

    Returns:
        float: DP score.
    """
    return G.degree[i] * G.degree[j]


def jaccard_coefficient(G, i, j):
    """
    Computes the Jaccard Coefficient (JC) for a given node pair (i, j).

    Parameters:
        G (networkx.Graph): The observed graph.
        i (int): Node i.
        j (int): Node j.

    Returns:
        float: JC score.
    """
    neighbors_i = set(G.neighbors(i))
    neighbors_j = set(G.neighbors(j))

    intersection = len(neighbors_i & neighbors_j)
    union = len(neighbors_i | neighbors_j)

    return intersection / union if union != 0 else 0


def shortest_path_score(G, i, j, epsilon=1e-6):
    """
    Computes the Shortest Path (SP) predictor score for a pair of nodes (i, j).

    Parameters:
        G (networkx.Graph): The observed graph.
        i (int): First node.
        j (int): Second node.
        epsilon (float, optional): Small random noise added to avoid ties (default: 1e-6).

    Returns:
        float: The SP score for (i, j).
    """
    try:
        sp_length = nx.shortest_path_length(G, source=i, target=j)
        return (1 / sp_length) + np.random.uniform(0, epsilon)
    except nx.NetworkXNoPath:
        return 0  # If no path exists, the score is 0.


def compute_auc(df, score_column, true_links):
    """
    Computes the AUC for a given score column in the dataframe.

    Parameters:
        df (pd.DataFrame): A dataframe containing columns ['i', 'j', 'score'].
        score_column (str): The name of the score column to use.
        true_links (set): A set of true edges (i, j) in the original graph.

    Returns:
        float: The computed AUC value.
    """
    # Assign ground truth labels: 1 for existing edges, 0 for non-existing
    df["true_label"] = df.apply(
        lambda row: 1 if (row["i"], row["j"]) in true_links else 0, axis=1
    )

    # Compute AUC using sklearn's roc_auc_score
    auc = roc_auc_score(df["true_label"], df[score_column])
    return auc


def generate_observed_graph(G, f):
    """
    Generates an observed graph G' by removing a fraction f of edges from G.

    Parameters:
        G (networkx.Graph): The original full graph.
        f (float): Fraction of edges to keep.

    Returns:
        networkx.Graph: The observed graph G' with missing edges.
        set: The set of removed edges.
    """
    G_prime = G.copy()
    edges = list(G.edges)
    np.random.shuffle(edges)  # Shuffle edges for randomness

    num_remove = int((1 - f) * len(edges))  # Number of edges to remove
    removed_edges = set(edges[:num_remove])  # Select edges to remove

    G_prime.remove_edges_from(removed_edges)  # Remove the selected edges

    return G_prime, removed_edges


def generate_prediction_table(G_prime, G, removed_edges):
    """
    Generates a table of link prediction scores for potential missing links.

    Parameters:
        G_prime (networkx.Graph): The observed graph.
        G (networkx.Graph): The original full graph.
        removed_edges (set): The set of true missing edges.

    Returns:
        pd.DataFrame: DataFrame containing node pairs and their predictor scores.
    """
    all_possible_edges = set(nx.non_edges(G_prime))  # Get all possible edges
    potential_links = list(all_possible_edges)  # Convert to list

    data = []
    for i, j in potential_links:
        dp_score = degree_product(G_prime, i, j)
        jc_score = jaccard_coefficient(G_prime, i, j)
        sp_score = shortest_path_score(G_prime, i, j)

        data.append((i, j, dp_score, jc_score, sp_score))

    df = pd.DataFrame(data, columns=["i", "j", "DP", "JC", "SP"])
    return df


def evaluate_link_prediction(G, f_values, num_trials=10):
    """
    Evaluates AUC scores for the three predictors over different f values.

    Parameters:
        G (networkx.Graph): The original graph.
        f_values (list): List of f values to evaluate.
        num_trials (int): Number of trials for averaging AUC.

    Returns:
        dict: Dictionary with f-values as keys and AUC scores as values.
    """
    auc_results = {"DP": [], "JC": [], "SP": []}

    for f in f_values:
        dp_scores, jc_scores, sp_scores = [], [], []

        for _ in range(num_trials):
            G_prime, removed_edges = generate_observed_graph(G, f)
            df = generate_prediction_table(G_prime, G, removed_edges)

            dp_auc = compute_auc(df, "DP", removed_edges)
            jc_auc = compute_auc(df, "JC", removed_edges)
            sp_auc = compute_auc(df, "SP", removed_edges)

            dp_scores.append(dp_auc)
            jc_scores.append(jc_auc)
            sp_scores.append(sp_auc)

        # Store averaged AUC
        auc_results["DP"].append(np.mean(dp_scores))
        auc_results["JC"].append(np.mean(jc_scores))
        auc_results["SP"].append(np.mean(sp_scores))

    return auc_results


def plot_accuracy_curves(f_values, auc_results, network_name):
    """
    Plots accuracy curves (AUC vs. f) for the three predictors.

    Parameters:
        f_values (list): List of f values.
        auc_results (dict): Dictionary containing AUC values for each predictor.
        network_name (str): Name of the network.
    """
    plt.figure(figsize=(8, 6))
    plt.plot(f_values, auc_results["DP"], label="Degree Product (DP)", marker="o")
    plt.plot(f_values, auc_results["JC"], label="Jaccard Coefficient (JC)", marker="s")
    plt.plot(f_values, auc_results["SP"], label="Shortest Path (SP)", marker="^")
    plt.axhline(y=0.5, color="k", linestyle="--", label="Random Guess (AUC = 0.5)")

    plt.xlabel("Fraction of Observed Edges (f)")
    plt.ylabel("AUC")
    plt.title(f"Accuracy of Predictors on {network_name}")
    plt.legend()
    plt.grid()
    plt.show()


def plot_roc_curves(df, removed_edges, network_name):
    """
    Plots ROC curves for all three predictors.

    Parameters:
        df (pd.DataFrame): The dataframe containing prediction scores.
        removed_edges (set): The set of true missing edges.
        network_name (str): Name of the network.
    """
    plt.figure(figsize=(8, 6))

    for predictor in ["DP", "JC", "SP"]:
        df["true_label"] = df.apply(
            lambda row: 1 if (row["i"], row["j"]) in removed_edges else 0, axis=1
        )
        fpr, tpr, _ = roc_curve(df["true_label"], df[predictor])
        plt.plot(
            fpr,
            tpr,
            label=f"{predictor} (AUC = {compute_auc(df, predictor, removed_edges):.2f})",
        )

    plt.plot([0, 1], [0, 1], "k--", label="Random Guess")
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.title(f"ROC Curves for {network_name} (f = 0.8)")
    plt.legend()
    plt.grid()
    plt.show()


def plot_accuracy_curves_plotnine(f_values, auc_results, network_name):
    """
    Plots accuracy curves (AUC vs. f) for the three predictors using plotnine.

    Parameters:
        f_values (list): List of f values.
        auc_results (dict): Dictionary containing AUC values for each predictor.
        network_name (str): Name of the network.
    """
    auc_df = pd.DataFrame(auc_results)
    auc_df["f"] = f_values

    # Melt the dataframe for ggplot
    auc_df_melted = auc_df.melt(id_vars="f", var_name="Predictor", value_name="AUC")

    # rename the predictor names to "Degree Product", "Jaccard Coefficient", "Shortest Path"
    auc_df_melted["Predictor"] = auc_df_melted["Predictor"].replace(
        {"DP": "Degree Product", "JC": "Jaccard Coefficient", "SP": "Shortest Path"}
    )
    auc_df_melted["Predictor"] = pd.Categorical(
        auc_df_melted["Predictor"],
        categories=["Degree Product", "Jaccard Coefficient", "Shortest Path"],
        ordered=True,
    )

    # Plot using ggplot
    accuracy_plot = (
        ggplot(auc_df_melted, aes(x="f", y="AUC", color="Predictor"))
        + geom_line(size=1.2)
        + geom_line(aes(x="f", y=0.5), linetype="dashed", color="black")
        + geom_point(size=2)
        + labs(
            x="Fraction of Observed Edges (f)",
            y="AUC",
            title=f"Accuracy of Predictors on {network_name}",
        )
        + theme_minimal()
        + scale_color_brewer(type="qual", palette="Set1")
        + theme(legend_position="none")
    )

    return accuracy_plot


def plot_roc_curves_plotnine(df, removed_edges, network_name):
    """
    Plots ROC curves for all three predictors using plotnine.

    Parameters:
        df (pd.DataFrame): The dataframe containing prediction scores.
        removed_edges (set): The set of true missing edges.
        network_name (str): Name of the network.
    """
    roc_df = df.copy()
    roc_df["true_label"] = roc_df.apply(
        lambda row: 1 if (row["i"], row["j"]) in removed_edges else 0, axis=1
    )

    roc_list = []  # Store individual DataFrames in a list

    for predictor in ["DP", "JC", "SP"]:
        fpr, tpr, _ = roc_curve(roc_df["true_label"], roc_df[predictor])
        roc_list.append(pd.DataFrame({"fpr": fpr, "tpr": tpr, "Predictor": predictor}))

    # Concatenate all stored DataFrames at once
    roc_data = pd.concat(roc_list, ignore_index=True)

    # Rename predictor labels for readability
    roc_data["Predictor"] = roc_data["Predictor"].replace(
        {"DP": "Degree Product", "JC": "Jaccard Coefficient", "SP": "Shortest Path"}
    )
    roc_data["Predictor"] = roc_data["Predictor"].astype("category")

    # Create ROC plot
    roc_plot = (
        ggplot(roc_data, aes(x="fpr", y="tpr", color="Predictor"))
        + geom_line(size=1.2)
        # + geom_point(size=2)
        + geom_abline(linetype="dashed")
        + labs(
            x="False Positive Rate",
            y="True Positive Rate",
            title=f"ROC Curves for {network_name} (f = 0.8)",
        )
        + theme_minimal()
        + scale_color_brewer(type="qual", palette="Set1")
        + theme(legend_position="right")
    )

    return roc_plot

In [17]:
f_values = np.arange(0.1, 1.0, 0.1)  # Test different f values

# Evaluate performance on malaria network
malaria_auc_results = evaluate_link_prediction(malaria_G, f_values)
bod_auc_results = evaluate_link_prediction(bod_G, f_values)

# Plot accuracy curves
# plot_accuracy_curves(f_values, malaria_auc_results, "Malaria Network")

# Generate prediction table for f = 0.8
G_prime_80, removed_edges_80 = generate_observed_graph(malaria_G, 0.8)
G_prime_80_bod, removed_edges_80_bod = generate_observed_graph(bod_G, 0.8)

# Generate prediction tables 
df_80 = generate_prediction_table(G_prime_80, malaria_G, removed_edges_80)
df_80_bod = generate_prediction_table(G_prime_80_bod, bod_G, removed_edges_80_bod)

# Plot ROC curves
# plot_roc_curves(df_80, removed_edges_80, "Malaria Network")

In [19]:
plot = plot_accuracy_curves_plotnine(f_values, malaria_auc_results, "Malaria Network")
plot.save("figures/malaria_link_prediction_accuracy.png")


roc_plot = plot_roc_curves_plotnine(df_80, removed_edges_80, "Malaria Network")
roc_plot.save("figures/malaria_roc_curves.png")

plot_bod = plot_accuracy_curves_plotnine(f_values, bod_auc_results, "Board Of Directors Network")
plot_bod.save("figures/bod_link_prediction_accuracy.png")

roc_plot_bod = plot_roc_curves_plotnine(
    df_80_bod, removed_edges_80_bod, "Board Of Directors Network"
)
roc_plot_bod.save("figures/bod_roc_curves.png")

/opt/anaconda3/envs/networks/lib/python3.12/site-packages/plotnine/ggplot.py:615: PlotnineWarning: Saving 6.4 x 4.8 in image.
/opt/anaconda3/envs/networks/lib/python3.12/site-packages/plotnine/ggplot.py:616: PlotnineWarning: Filename: figures/malaria_link_prediction_accuracy.png
/opt/anaconda3/envs/networks/lib/python3.12/site-packages/plotnine/ggplot.py:615: PlotnineWarning: Saving 6.4 x 4.8 in image.
/opt/anaconda3/envs/networks/lib/python3.12/site-packages/plotnine/ggplot.py:616: PlotnineWarning: Filename: figures/malaria_roc_curves.png
/opt/anaconda3/envs/networks/lib/python3.12/site-packages/plotnine/ggplot.py:615: PlotnineWarning: Saving 6.4 x 4.8 in image.
/opt/anaconda3/envs/networks/lib/python3.12/site-packages/plotnine/ggplot.py:616: PlotnineWarning: Filename: figures/bod_link_prediction_accuracy.png
/opt/anaconda3/envs/networks/lib/python3.12/site-packages/plotnine/ggplot.py:615: PlotnineWarning: Saving 6.4 x 4.8 in image.
/opt/anaconda3/envs/networks/lib/python3.12/site-pac

(b) (40 pts extra credit) Supervised link prediction.
- Implement a stacking model that uses the DP, JC, and SP predictors as level-0 algorithms, and uses either a random forest (RF) or logistic regression (LR) algorithm (your choice) as the supervised meta-learner for the level-1 algorithm.
- To evaluate the stacked model’s accuracy, we will need to ‘observe’ the network in a 2-step process because we need a training/test split to train the supervised algorithm, but we also need held-out data to evaluate the trained algorithm.
    - Recall that $G' = (V, E')$ where $E'$ is the observed set of edges (each observed with probability $f$). Using the observed network $G'$, create the validation data set by calculating the level-0 algorithms for each pair $i, j \in X$, the set of possible missing links in $G'$. (Recall that $Y = E − E'$ tells you which pairs $i, j \in X$ are missing links.) Store the validation data as matrix $S$.
    - Now deﬁne the training graph $G'' = (V, E'')$, where $E''$ is a fraction $f$ sample of the observed edges. Create the training data set by selecting uniformly at random $c$ pairs from $Y' = E' − E''$ (true positives for $G''$) and $c$ pairs from $X' − Y'$ (true negatives for $G''$); then run the level-0 algorithms for each of these $2c$ pairs, using $G''$. Store the training data as matrix $T$.
    - Train the level-1 model to predict, on the basis of the training data $T$, whether $i, j \in Y'$.
    - Apply the trained model to the validation data $S$, and evaluate its predictions relative to $Y$.
- Choose just one of the two networks you’ve used so far and make one nice ﬁgure for each network plotting AUC values for $f = {0.5, 0.6, 0.7, 0.8, 0.9}$. Add to this ﬁgure the corresponding AUC values for the DP, JC, and SP algorithms from question 2a, and legend indicating which is which
- For $f = 0.8$, make a nice ﬁgure showing the three full ROC curves for DP, JC, and SP, and add a fourth line showing the ROC curve for the stacked model.
- Discuss: (i) To what extent does the stacked model’s AUC improve upon, or not, the level-0 model AUCs? And, (ii) to what extent does the stacked model’s ROC curve improve upon, or not, the level-0 model ROC curves?

Hint: for the training data, $c = 1000$ is a good number. However, for modest-sized networks, there may not be so many true positives; in this case, we upsample the true positives by sampling with replacement until we have $c = 1000$ examples.